# 6GBench MCQ Evaluation with Ollama

## Before you run
Make sure Ollama is already running and that your target model is available:

```bash
ollama list
ollama pull llama3.2:1b
```

If your environment provides `OLLAMA_HOST`, the notebook will use it automatically. Otherwise it falls back to `http://localhost:11434`.


In [21]:
import sys
!{sys.executable} -m pip install -q -U ollama requests tqdm
print("Installed required Python packages.")


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Installed required Python packages.


In [22]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import re
import statistics
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from ollama import Client
from tqdm.auto import tqdm

In [23]:
# === Configuration ===
DATA_DIR = Path("/home/jupyter-mferrag/data/6GBench/mcq_questions_only")
SUMMARY_MAX_TURNS = 12

# Pick a model that exists in `ollama list`
EVAL_MODEL = "smollm2:135m-instruct-q4_0"

# Use session-provided host if available, else localhost
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")

# Pass@k config
PASSK_TASKS = {"T2", "T9", "T12", "T19", "T20", "T26", "T30"}
PASSK_KS = [3, 5]
PASSK_TEMPERATURE = 0.7

print("DATA_DIR:", DATA_DIR)
print("EVAL_MODEL:", EVAL_MODEL)
print("OLLAMA_HOST:", OLLAMA_HOST)

DATA_DIR: /home/jupyter-mferrag/data/6GBench/mcq_questions_only
EVAL_MODEL: smollm2:135m-instruct-q4_0
OLLAMA_HOST: http://localhost:11434


In [24]:
# Full task registry
TASKS: List[Tuple[str, str, str]] = [
  ("T1", "Intent Feasibility Assessment", "Given a mission and a 6G intent message, determine whether the intent is feasible under current and near-future network, environmental, and policy constraints. Identify the minimal safe adjustments (e.g., speed, route, slice, autonomy level, sensing configuration) needed to satisfy constraints while preserving mission objectives as much as possible."),
  ("T2", "Intent Conflict Resolution", "Resolve conflicts between the operator/mission intent and network or safety policy (e.g., airspace, security, energy, SLA). Decide whether to reject, modify, or conditionally approve the intent, and specify concrete policy-aligned adjustments that balance mission goals and compliance."),
  ("T3", "Intent Drift Detection", "Detect subtle changes in mission or network intent over time (e.g., updated priorities, new safety requirements, shifted QoS targets) by comparing past and current intents and behavior. Decide whether the drift is benign, problematic, or requires renegotiation or clarification with other agents or controllers."),
  ("T4", "Slice Selection Reasoning", "Given mission requirements and current network telemetry, choose between URLLC, eMBB, or a hybrid slice (or slice configuration) with explicit justification. Trade off latency, reliability, throughput, and robustness, and explain why alternative slices are less appropriate in the given context."),
  ("T5", "Slice Switching Decision", "Decide whether to switch, maintain, or augment the current network slice when performance degrades, considering stability, hysteresis, mission criticality, and switching overheads. Prefer decisions that avoid unnecessary oscillations while still preventing SLA violation or safety risk."),
  ("T6", "Slice Fairness vs Safety", "Resolve contention for slice resources among multiple agents or swarm members when their demands conflict. Balance fairness, priority levels, and safety margins, possibly degrading some agents more than others, while ensuring global mission safety and compliance with policies."),
  ("T7", "Compute Placement Decision", "Choose where to execute AI inference or other compute tasks (onboard, edge, peer, or cloud) under latency, bandwidth, energy, model quality, and trust constraints. Justify placement by considering dynamic network conditions, SLA requirements, and potential failure modes of each location."),
  ("T8", "Graceful Degradation under Edge Overload", "When edge resources become overloaded or unstable, decide how to gracefully degrade autonomy or service quality before SLAs are violated. Select which functions to simplify, slow down, or disable while preserving safety-critical behavior and mission viability as long as possible."),
  ("T9", "Trust-Aware Offloading", "Evaluate whether to offload tasks or data to edge or third-party compute resources based on trust, security, and policy constraints. Decide when to reject offloading, use partial offloading, or require additional safeguards (e.g., encryption, sandboxing) despite potential performance benefits."),
  ("T10", "SLA Violation Prediction", "Predict future SLA violations using early network and system signals such as latency trends, jitter, loss, throughput, edge load, and mission dynamics. Distinguish between transient fluctuations and meaningful trends, and indicate when preemptive mitigation is required to avoid imminent violation."),
  ("T11", "Preemptive Autonomy Downgrade", "Before any actual failure or SLA violation occurs, decide when and how to safely downgrade autonomy or functionality based on predicted risk. Choose specific behaviors or capabilities to limit, explaining why the downgrade is justified and how it preserves overall mission safety and compliance."),
  ("T12", "Conservative Continuation Decision", "Under uncertainty about network, sensing, or environment, decide whether to continue the mission in a conservative mode or pause/abort. Weigh incomplete or noisy evidence, risk to safety and SLA, and mission criticality, preferring nuanced partial continuation when strictly safe and justifiable."),
  ("T13", "Swarm-Level Slice Negotiation", "Coordinate slice allocation across multiple agents or swarm members with competing demands and priorities. Decide how to negotiate and partition slice resources over time, potentially reallocating or renegotiating as conditions change while maintaining global mission performance and fairness."),
  ("T14", "Scheduler Reconfiguration Adaptation", "When the underlying AI or network scheduler is reconfigured, updated, or replaced, maintain decision consistency and mission safety. Detect behavioral changes introduced by the new scheduler and adapt policies or intents so that overall system behavior remains coherent and policy-compliant."),
  ("T15", "Decision Consistency under Replanning", "Ensure that decisions across multiple planning cycles or turns remain logically consistent with prior commitments, unless new evidence necessitates a change. Avoid contradictory or oscillatory decisions, and when changes are required, justify them with explicit reference to updated context or constraints."),
  ("T16", "Network-Exposed Compute Marketplace", "Decide whether, when, and how to expose operator edge/cloud compute resources as a marketplace to third parties under current load, SLAs, and policies. Determine pricing, admission, and allocation strategies that protect critical network services while extracting value from idle capacity."),
  ("T17", "Network-Knowledge RAG Augmentation", "Decide what network telemetry, logs, and knowledge to expose to Retrieval-Augmented Generation systems to enhance agent reasoning, under privacy, security, and latency constraints. Balance informativeness against overhead and policy limits, selecting only the most relevant and safe signals."),
  ("T18", "AI Agent Identity & Onboarding", "Authorize, authenticate, and register AI agents (device- or network-hosted) and decide how they are represented in identity and access control systems. Define identity mapping, credentials, and onboarding flows that respect policy, security, and interoperability requirements over the agent lifecycle."),
  ("T19", "AI Agent Interoperability & Federation", "Resolve compatibility and data-sharing decisions when multiple AI agents from different domains, operators, or networks must collaborate. Decide protocols, translation layers, and data access policies that enable coordination while respecting trust boundaries, privacy, and regulatory constraints."),
  ("T20", "Agent-to-Agent Communication Management", "Decide routing, QoS, and security policies for horizontal traffic between AI agents, whether direct or via network relays. Prioritize flows, select paths or slices, and enforce encryption or isolation as needed to meet latency, reliability, and security goals under dynamic network conditions."),
  ("T21", "Device-Network Task Offload Arbitration", "Choose whether and how to offload AI or compute tasks from a device to edge, peer, or cloud resources given latency, energy, model capability, and trust constraints. Consider partial offloading, model selection, and fallback strategies, ensuring that decisions remain robust to network fluctuations."),
  ("T22", "Federated / Collaborative Learning Orchestration", "Decide when and how to schedule federated or collaborative training and model updates across devices and edge nodes. Respect privacy, regulatory constraints, bandwidth limits, and device heterogeneity, and choose update frequencies and participant sets that balance model quality and resource usage."),
  ("T23", "Network-Assisted Digital Twin Control", "Determine how the network should provide sensing, telemetry, and control channels to maintain accurate and actionable real-time digital twins. Decide update rates, data fidelity, and control loop configurations that keep the twin synchronized without overloading the network or violating SLAs."),
  ("T24", "Sensing-Enhanced Decisioning (ISAC)", "Choose which sensing streams (e.g., radar, RF, vision, telemetry) and fusion strategies the network should deliver to agents for time-sensitive perception and decision tasks. Trade off sensing accuracy, bandwidth, latency, and robustness, and adapt the sensing configuration as conditions evolve."),
  ("T25", "AI-Agent-based Disaster / Public-Safety Coordination", "Coordinate multiple AI agents and UAVs during disaster or public-safety scenarios, deciding slice allocation, sensing priorities, and escalation paths. Balance competing mission goals such as search and rescue, damage assessment, and communication support under extreme and uncertain conditions."),
  ("T26", "Trust-Aware Third-Party Agent Exposure", "Decide what level of data, APIs, and compute resources to expose to third-party agents based on trust scores, regulation, and user consent. Enforce differentiated access and isolation policies, and adapt exposure in response to observed behavior, anomalies, or changing regulatory constraints."),
  ("T27", "Agent Lifecycle & Management", "Decide lifecycle operations for agents, including instantiation, scaling, migration, upgrade, and retirement, under operator policy and SLA constraints. Coordinate these operations with network load, security posture, and mission demands to avoid service disruption or policy violations."),
  ("T28", "6G Model Training-as-a-Service Decision", "Decide when to accept or reject customer requests for network-facilitated model training (e.g., LLM fine-tuning) given resource availability, privacy requirements, and QoS impact. Determine appropriate training configurations, isolation levels, and scheduling so that core network services remain protected."),
  ("T29", "Immersive/AR Resource Prioritization", "Allocate slices and edge resources to multi-modal immersive or XR/AR sessions, balancing throughput, latency, stability, and fairness across users and applications. Handle contention by prioritizing critical interactions and gracefully degrading less critical modalities when resources are constrained."),
  ("T30", "Network Security Detection & Response Automation", "When AI-driven monitoring flags potential attacks or anomalies, decide automated detection, isolation, mitigation, and recovery actions in the network. Balance swift containment with false-positive risk, and choose responses that preserve critical services and safety while minimizing collateral impact."),
]
TASK_MAP: Dict[str, Tuple[str, str]] = {tid: (name, desc) for tid, name, desc in TASKS}
print("Loaded", len(TASKS), "tasks.")

Loaded 30 tasks.


In [25]:
@dataclass
class MCQQuestion:
    task_id: str
    task_name: str
    source_turn: int
    question: str
    options: Dict[str, str]
    correct: str
    reason: str
    rationale_tag: str
    difficulty: str

In [26]:
_OLLAMA_CLIENT: Optional[Client] = None

def get_ollama_client() -> Client:
    global _OLLAMA_CLIENT
    if _OLLAMA_CLIENT is None:
        _OLLAMA_CLIENT = Client(host=OLLAMA_HOST)
    return _OLLAMA_CLIENT

def list_ollama_models() -> List[str]:
    client = get_ollama_client()
    resp = client.list()

    names: List[str] = []
    if isinstance(resp, dict):
        for m in resp.get("models", []):
            name = m.get("model") or m.get("name")
            if name:
                names.append(name)
    else:
        try:
            for m in resp.models:
                name = getattr(m, "model", None) or getattr(m, "name", None)
                if name:
                    names.append(name)
        except Exception:
            pass

    return sorted(set(names))

def ensure_model_available(model_name: str) -> None:
    names = list_ollama_models()
    if model_name not in names:
        raise ValueError(
            f"Model '{model_name}' not found in Ollama. Available models: {names}\n"
            f"Run `ollama pull {model_name}` in a terminal first."
        )

def unload_model(model_name: Optional[str] = None) -> None:
    """
    Unload an Ollama model from memory.

    - If model_name is provided: request unload for that model.
    - If model_name is None: just run local Python garbage collection.
    """
    client = get_ollama_client()

    if not model_name:
        gc.collect()
        print("No model_name provided; ran local garbage collection only.")
        return

    try:
        client.generate(
            model=model_name,
            prompt="",
            keep_alive=0,
        )
        gc.collect()
        print(f"Unloaded model from Ollama memory: {model_name}")
    except Exception as e:
        gc.collect()
        print(f"Warning: failed to unload model '{model_name}': {e}")

In [27]:
print("Ollama models visible from this notebook:")
models = list_ollama_models()
print(models if models else "No models found.")
if models and EVAL_MODEL not in models:
    print(f"Warning: EVAL_MODEL={EVAL_MODEL!r} is not currently installed.")

Ollama models visible from this notebook:
['mistral-medium-3.5:latest', 'smollm2:135m-instruct-q4_0']


In [28]:
import subprocess
import threading
import time
from typing import Optional, List

def _read_gpu_memory_used_gb(gpu_index: int = 0) -> Optional[float]:
    """
    Returns current GPU memory used in GB for one GPU via nvidia-smi.
    Returns None if nvidia-smi is unavailable or parsing fails.
    """
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                f"--id={gpu_index}",
                "--query-gpu=memory.used",
                "--format=csv,noheader,nounits",
            ],
            stderr=subprocess.DEVNULL,
        ).decode().strip()

        # Example output: "7423"
        used_mb = float(out.splitlines()[0].strip())
        return used_mb / 1024.0
    except Exception:
        return None


def measure_peak_vram_during_call(fn, poll_interval_s: float = 0.05, gpu_index: int = 0):
    """
    Runs fn() while sampling GPU memory.
    Returns: (fn_result, peak_vram_gb)

    peak_vram_gb is the max observed GPU memory.used during the call.
    This is an estimate based on polling.
    """
    stop_flag = {"stop": False}
    samples: List[float] = []

    def poller():
        while not stop_flag["stop"]:
            val = _read_gpu_memory_used_gb(gpu_index=gpu_index)
            if val is not None:
                samples.append(val)
            time.sleep(poll_interval_s)

    t = threading.Thread(target=poller, daemon=True)
    t.start()
    try:
        result = fn()
    finally:
        stop_flag["stop"] = True
        t.join(timeout=1.0)

    peak_vram_gb = max(samples) if samples else None
    return result, peak_vram_gb

In [29]:
def get_turns(ep: Dict[str, Any]) -> List[Dict[str, Any]]:
    return ep.get("dialogue", [])

def extract_min_context(ep: Dict[str, Any]) -> Dict[str, Any]:
    init_state = ep.get("initial_state", {})
    env = init_state.get("env") or {}
    airspace = init_state.get("airspace") or {}
    uav = init_state.get("uav") or {}
    policy = init_state.get("policy") or {}
    sensors = uav.get("sensors") or {}
    return {
        "env": env,
        "airspace": {"alt_bounds": airspace.get("alt_bounds"), "geofence": airspace.get("geofence")},
        "uav": {
            "pose": uav.get("pose"),
            "speed_mps": uav.get("speed_mps"),
            "battery_pct": (uav.get("energy") or {}).get("battery_pct"),
            "sensors": sensors,
            "payloads": uav.get("payloads", []),
        },
        "policy": policy,
        "success": ep.get("success"),
    }

def summarize_episode_for_prompt(ep: Dict[str, Any], max_turns: int = 12) -> str:
    ctx = extract_min_context(ep)
    parts: List[str] = []
    parts.append("Initial context:")
    parts.append(f"- env: {ctx['env']}")
    parts.append(f"- airspace: {ctx['airspace']}")
    parts.append(f"- uav: {ctx['uav']}")
    parts.append(f"- policy: {ctx['policy']}")
    parts.append(f"- success: {ctx['success']}")
    parts.append("")
    parts.append("Dialogue trace (truncated):")
    for t in get_turns(ep)[:max_turns]:
        turn_no = t.get("turn")
        speaker = t.get("speaker")
        intent = t.get("intent")
        acts = t.get("actions", [])
        act_summaries = []
        for a in acts:
            if a.get("type") == "mcp":
                arg_keys = list((a.get("args") or {}).keys())
                act_summaries.append(f"mcp:{a.get('name')} args_keys={arg_keys}")
            elif a.get("type") == "a2a":
                act_summaries.append(f"a2a:{a.get('task')} to={a.get('to')}")
        obs = t.get("obs", [])
        obs_summaries = []
        for o in obs[:3]:
            if "tool" in o:
                res = o.get("result", {})
                status = res.get("status") if isinstance(res, dict) else None
                obs_summaries.append(f"tool:{o.get('tool')} status={status}")
            elif "task" in o:
                obs_summaries.append(f"a2a_resp:{o.get('task')} status={o.get('status')}")
        net = t.get("net") or {}
        net_s = {
            "slice": net.get("slice"),
            "lat_ms": net.get("lat_ms"),
            "jitter_ms": net.get("jitter_ms"),
            "loss_pct": net.get("loss_pct"),
            "throughput_mbps": net.get("throughput_mbps"),
            "edge_load": net.get("edge_load"),
        }
        parts.append(
            f"- turn={turn_no} speaker={speaker} intent={intent} "
            f"actions={act_summaries} obs={obs_summaries} net={net_s}"
        )
    return "\n".join(parts)

def build_eval_messages(task_id: str, episode_summary: str, question: Dict[str, Any]) -> List[Dict[str, str]]:
    task_name, task_def = TASK_MAP[task_id]
    q_text = question["question"]
    options = question["options"]

    user_content: List[str] = []
    user_content.append("TARGET TASK")
    user_content.append(f"TASK_ID: {task_id}")
    user_content.append(f"TASK_NAME: {task_name}")
    user_content.append(f"DEFINITION: {task_def}")
    user_content.append("")
    user_content.append("EPISODE SUMMARY")
    user_content.append(episode_summary)
    user_content.append("")
    user_content.append("QUESTION")
    user_content.append(q_text)
    user_content.append("")
    user_content.append("OPTIONS")
    for k in ["A", "B", "C", "D"]:
        user_content.append(f"{k}: {options[k]}")
    user_content.append("")
    user_content.append(
        'Respond ONLY with valid JSON of the form {"answer": "A/B/C/D"} where answer is one of "A", "B", "C", or "D".'
    )

    system_msg = (
        "You are an expert 6G network AI agent evaluator. "
        "You will answer a multiple-choice question (A/B/C/D) about a UAV mission episode. "
        "Use the episode context and the target task definition to choose the best answer. "
        'You MUST respond with a single JSON object of the form {"answer": "A"}, '
        '{"answer": "B"}, {"answer": "C"}, or {"answer": "D"}.'
    )

    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": "\n".join(user_content)},
    ]

def parse_mcq_answer_from_json(raw: Any) -> Optional[str]:
    if isinstance(raw, dict):
        obj = raw
    else:
        try:
            obj = json.loads(raw)
        except Exception:
            obj = None

    if isinstance(obj, dict):
        for key in ["answer", "choice", "label", "option"]:
            val = obj.get(key)
            if isinstance(val, str) and val.strip() in {"A", "B", "C", "D"}:
                return val.strip()

    text = raw if isinstance(raw, str) else str(raw)
    m = re.search(r'"answer"\s*:\s*"([ABCD])"', text)
    if m:
        return m.group(1)
    m2 = re.search(r"\b([ABCD])\b", text)
    if m2:
        return m2.group(1)
    return None

In [30]:
def load_episode_mcq_pairs(data_dir: Path) -> List[Tuple[str, Dict[str, Any], Dict[str, Any]]]:
    episodes: Dict[str, Path] = {}
    mcqs: Dict[str, Path] = {}

    for p in data_dir.glob("*.episode.json"):
        base = p.name.replace(".episode.json", "")
        episodes[base] = p

    for p in data_dir.glob("*.mcq.json"):
        base = p.name.replace(".mcq.json", "")
        mcqs[base] = p

    common_keys = sorted(set(episodes.keys()) & set(mcqs.keys()))
    pairs: List[Tuple[str, Dict[str, Any], Dict[str, Any]]] = []

    for key in common_keys:
        with episodes[key].open("r", encoding="utf-8") as f_ep:
            ep = json.load(f_ep)
        with mcqs[key].open("r", encoding="utf-8") as f_q:
            mcq = json.load(f_q)
        episode_id = mcq.get("episode_id", key)
        pairs.append((episode_id, ep, mcq))

    print(f"Found {len(pairs)} episode/MCQ pairs in {data_dir}")
    return pairs

def print_task_question_counts(data_dir: Path) -> Dict[str, int]:
    pairs = load_episode_mcq_pairs(data_dir)
    task_counts: Dict[str, int] = {}
    total_questions = 0

    for _, _, mcq in pairs:
        for q in mcq.get("questions", []):
            task_id = q.get("task_id", "UNKNOWN")
            task_counts[task_id] = task_counts.get(task_id, 0) + 1
            total_questions += 1

    print("\nQuestion count per task:")
    print("-" * 60)
    for task_id, count in sorted(task_counts.items()):
        task_name, _ = TASK_MAP.get(task_id, ("?", ""))
        print(f"{task_id:>3} | {task_name:<45} | {count:5d}")
    print("-" * 60)
    print(f"TOTAL QUESTIONS: {total_questions}")
    return task_counts

In [31]:
def local_chat_debug(
    model: str,
    messages: List[Dict[str, str]],
    temperature: float = 0.2,
    max_tokens: int = 10000,
    seed: Optional[int] = None,
    gpu_index: int = 0,
    poll_interval_s: float = 0.05,
) -> Tuple[str, Dict[str, Any]]:
    """
    Run a local Ollama chat request and return:
      (completion_text, debug_info)

    Adds estimated peak VRAM by polling nvidia-smi during the request.
    """
    client = get_ollama_client()
    ensure_model_available(model)

    response_schema = {
        "type": "object",
        "properties": {
            "answer": {
                "type": "string",
                "enum": ["A", "B", "C", "D"]
            }
        },
        "required": ["answer"]
    }

    options = {
        "temperature": float(temperature),
        "num_predict": int(max_tokens),
    }
    if seed is not None:
        options["seed"] = int(seed)

    def do_chat():
        return client.chat(
            model=model,
            messages=messages,
            stream=False,
            options=options,
            format=response_schema,
        )

    resp, peak_vram_gb = measure_peak_vram_during_call(
        do_chat,
        poll_interval_s=poll_interval_s,
        gpu_index=gpu_index,
    )

    def _get(x, key, default=None):
        if isinstance(x, dict):
            return x.get(key, default)
        return getattr(x, key, default)

    message_obj = _get(resp, "message", {})
    if isinstance(message_obj, dict):
        completion = message_obj.get("content", "")
    else:
        completion = getattr(message_obj, "content", "")

    total_duration = _get(resp, "total_duration")
    load_duration = _get(resp, "load_duration")
    prompt_eval_count = _get(resp, "prompt_eval_count")
    prompt_eval_duration = _get(resp, "prompt_eval_duration")
    eval_count = _get(resp, "eval_count")
    eval_duration = _get(resp, "eval_duration")
    done_reason = _get(resp, "done_reason")
    created_at = _get(resp, "created_at")
    model_used = _get(resp, "model", model)

    latency_ms = (total_duration / 1e6) if total_duration is not None else None

    throughput_toks_per_s = None
    if eval_count is not None and eval_duration not in (None, 0):
        throughput_toks_per_s = float(eval_count) / (float(eval_duration) / 1e9)

    debug_info = {
        "model": model_used,
        "created_at": created_at,
        "done_reason": done_reason,
        "latency_ms": latency_ms,
        "load_duration_ms": (load_duration / 1e6) if load_duration is not None else None,
        "prompt_eval_count": prompt_eval_count,
        "prompt_eval_duration_ms": (prompt_eval_duration / 1e6) if prompt_eval_duration is not None else None,
        "eval_count": eval_count,
        "eval_duration_ms": (eval_duration / 1e6) if eval_duration is not None else None,
        "throughput_toks_per_s": throughput_toks_per_s,
        "used_max_new_tokens": max_tokens,
        "peak_vram_gb": peak_vram_gb,
        "input_tokens": prompt_eval_count,
        "output_tokens": eval_count,
    }

    return completion.strip(), debug_info

In [32]:
def eval_model_on_dir(
    data_dir: Path,
    model: str,
    max_pairs: Optional[int] = None,
    seed_base: Optional[int] = 42,
) -> Dict[str, Any]:
    pairs = load_episode_mcq_pairs(data_dir)
    if max_pairs is not None:
        pairs = pairs[:max_pairs]

    total_questions_all = sum(len(mcq.get("questions", [])) for _, _, mcq in pairs)
    avg_q_per_episode = total_questions_all / len(pairs) if pairs else 0.0
    print(
        f"Total episodes: {len(pairs)} | "
        f"Total questions (all episodes): {total_questions_all} | "
        f"Avg questions/episode: {avg_q_per_episode:.2f}"
    )

    total = 0
    correct = 0
    per_task_stats: Dict[str, Dict[str, int]] = {}
    per_question_records: List[Dict[str, Any]] = []

    per_task_passk_stats: Dict[str, Dict[int, Dict[str, int]]] = {}
    overall_passk_stats: Dict[int, Dict[str, int]] = {k: {"total": 0, "hit": 0} for k in PASSK_KS}

    pbar = tqdm(total=len(pairs), desc="Eval episodes", unit="ep")

    for episode_id, ep, mcq in pairs:
        episode_summary = summarize_episode_for_prompt(ep, max_turns=SUMMARY_MAX_TURNS)
        questions = mcq.get("questions", [])

        for q in questions:
            task_id = q["task_id"]
            gold = q["correct"]
            messages = build_eval_messages(task_id, episode_summary, q)

            seed = None
            if seed_base is not None:
                s = f"{seed_base}:{episode_id}:{task_id}".encode("utf-8")
                seed = int(hashlib.sha256(s).hexdigest()[:8], 16)

            raw_text, debug_info = local_chat_debug(
                model=model,
                messages=messages,
                temperature=0.0,
                max_tokens=10000,
                seed=seed,
            )

            pred = parse_mcq_answer_from_json(raw_text)
            is_correct = int(pred == gold)

            total += 1
            correct += is_correct

            stats = per_task_stats.setdefault(task_id, {"total": 0, "correct": 0})
            stats["total"] += 1
            stats["correct"] += is_correct

            per_question_records.append({
                "episode_id": episode_id,
                "task_id": task_id,
                "task_name": q.get("task_name"),
                "difficulty": q.get("difficulty"),
                "pred": pred,
                "gold": gold,
                "correct_flag": is_correct,
                "raw_model_output": raw_text,
                "question": q.get("question"),
                "options": q.get("options"),
                "source_turn": q.get("source_turn"),
                "debug_info": {
                    "latency_ms": debug_info.get("latency_ms"),
                    "peak_vram_gb": debug_info.get("peak_vram_gb"),
                    "used_max_new_tokens": debug_info.get("used_max_new_tokens"),
                    "input_tokens": debug_info.get("input_tokens"),
                    "output_tokens": debug_info.get("output_tokens"),
                    "throughput_toks_per_s": debug_info.get("throughput_toks_per_s"),
                },
            })

            if task_id in PASSK_TASKS:
                task_passk = per_task_passk_stats.setdefault(task_id, {})
                for k_val in PASSK_KS:
                    k_stats = task_passk.setdefault(k_val, {"total": 0, "hit": 0})
                    k_stats["total"] += 1
                    overall_passk_stats[k_val]["total"] += 1

                    hit = False
                    for attempt in range(k_val):
                        attempt_seed = None
                        if seed_base is not None:
                            s_k = f"{seed_base}:{episode_id}:{task_id}:k{k_val}:a{attempt}".encode("utf-8")
                            attempt_seed = int(hashlib.sha256(s_k).hexdigest()[:8], 16)

                        raw_k, _ = local_chat_debug(
                            model=model,
                            messages=messages,
                            temperature=PASSK_TEMPERATURE,
                            max_tokens=10000,
                            seed=attempt_seed,
                        )

                        pred_k = parse_mcq_answer_from_json(raw_k)
                        if pred_k == gold:
                            hit = True
                            break

                    if hit:
                        k_stats["hit"] += 1
                        overall_passk_stats[k_val]["hit"] += 1

        incorrect = total - correct
        current_acc = correct / total if total else 0.0

        pbar.update(1)
        pbar.set_postfix(q=total, ok=correct, wrong=incorrect, acc=f"{current_acc:.3f}")
        tqdm.write(
            f"[running] episodes={pbar.n}/{pbar.total} | "
            f"questions={total} | overall_acc={current_acc:.3f}"
        )

    pbar.close()

    overall_acc = correct / total if total else 0.0
    per_task_acc = {
        tid: s["correct"] / s["total"] if s["total"] else 0.0
        for tid, s in per_task_stats.items()
    }

    per_task_passk: Dict[str, Dict[int, float]] = {}
    for tid, k_stats_dict in per_task_passk_stats.items():
        per_task_passk[tid] = {}
        for k_val, stats_k in k_stats_dict.items():
            per_task_passk[tid][k_val] = stats_k["hit"] / stats_k["total"] if stats_k["total"] else 0.0

    overall_passk: Dict[int, float] = {}
    for k_val, stats_k in overall_passk_stats.items():
        overall_passk[k_val] = stats_k["hit"] / stats_k["total"] if stats_k["total"] else 0.0

    return {
        "model": model,
        "overall_accuracy": overall_acc,
        "per_task_accuracy": per_task_acc,
        "overall_passk": overall_passk,
        "per_task_passk": per_task_passk,
        "total_questions": total,
        "records": per_question_records,
    }

In [33]:
def safe_mean_std(xs: List[Optional[float]]) -> Tuple[Optional[float], Optional[float]]:
    xs = [x for x in xs if x is not None]
    if not xs:
        return None, None
    if len(xs) == 1:
        return float(xs[0]), 0.0
    return float(statistics.mean(xs)), float(statistics.stdev(xs))

def profile_single_question_with_ollama(
    eval_model: str,
    data_dir: Path,
    repetitions: int = 20,
    max_tokens: int = 10000,
    seed_base: int = 42,
    txt_log_path: Path = Path("data/6GBench/diag_profile_summary.txt"),
    json_out_path: Path = Path("data/6GBench/diag_profile_detail.json"),
    csv_out_path: Path = Path("data/6GBench/table_vi_rows.csv"),
    latex_out_path: Path = Path("data/6GBench/table_vi_row.tex"),
) -> Dict[str, Any]:
    pairs = load_episode_mcq_pairs(data_dir)
    if len(pairs) == 0:
        raise RuntimeError(f"No episode/mcq pairs found in {data_dir}")

    episode_id, ep, mcq = pairs[0]
    q = mcq["questions"][0]
    task_id = q["task_id"]

    messages = build_eval_messages(
        task_id,
        summarize_episode_for_prompt(ep, SUMMARY_MAX_TURNS),
        q,
    )

    print("--- Diagnostic: prompt length (chars) ---")
    print(sum(len(m["content"]) for m in messages), "chars")

    ensure_model_available(eval_model)

    per_run = []
    for i in range(repetitions):
        seed = seed_base + i
        completion, debug = local_chat_debug(
            model=eval_model,
            messages=messages,
            temperature=0.0,
            max_tokens=max_tokens,
            seed=seed,
        )

        d = dict(debug or {})
        d.update({
            "run_index": i,
            "seed": seed,
            "pred": parse_mcq_answer_from_json(completion) if completion else None,
            "completion_preview": completion[:1000] if completion else "",
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        per_run.append(d)

        if (i + 1) % max(1, repetitions // 10) == 0 or i == repetitions - 1:
            print(f"[diag] completed run {i+1}/{repetitions}")

    latencies = [r.get("latency_ms") for r in per_run if r.get("latency_ms") is not None]
    peak_mem = [r.get("peak_vram_gb") for r in per_run if r.get("peak_vram_gb") is not None]
    throughputs = [r.get("throughput_toks_per_s") for r in per_run if r.get("throughput_toks_per_s") is not None]

    lat_mean, lat_std = safe_mean_std(latencies)
    mem_mean, mem_std = safe_mean_std(peak_mem)
    thr_mean, thr_std = safe_mean_std(throughputs)

    first = per_run[0] if per_run else {}
    summary = {
        "model": eval_model,
        "episode_id": episode_id,
        "task_id": task_id,
        "device": "ollama-server",
        "input_tokens": first.get("input_tokens"),
        "output_tokens": first.get("output_tokens"),
        "repetitions": repetitions,
        "latency_ms_mean": lat_mean,
        "latency_ms_std": lat_std,
        "peak_vram_gb_mean": mem_mean,
        "peak_vram_gb_std": mem_std,
        "throughput_toks_per_s_mean": thr_mean,
        "throughput_toks_per_s_std": thr_std,
        "model_param_count": None,
        "used_max_new_tokens": first.get("used_max_new_tokens"),
        "notes": "Ollama profiling via API metadata; VRAM and parameter count are not directly exposed in standard chat responses.",
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

    txt_log_path.parent.mkdir(parents=True, exist_ok=True)
    with txt_log_path.open("a", encoding="utf-8") as t:
        t.write("=" * 80 + "\n")
        t.write(f"Diagnostic profiling run: {summary['timestamp']}\n")
        t.write(f"Model: {summary['model']}\n")
        t.write(f"Device: {summary['device']}\n")
        t.write(f"Episode: {summary['episode_id']}  Task: {summary['task_id']}\n")
        t.write(f"Input tokens: {summary['input_tokens']}  Output tokens (example): {summary['output_tokens']}\n")
        t.write(f"Repetitions: {summary['repetitions']}\n\n")
        t.write(f"Latency (ms): {summary['latency_ms_mean']:.2f} ± {summary['latency_ms_std']:.2f}\n" if summary["latency_ms_mean"] is not None else "Latency (ms): N/A\n")
        t.write(f"Peak VRAM (GB): {summary['peak_vram_gb_mean']:.3f} ± {summary['peak_vram_gb_std']:.3f}\n" if summary["peak_vram_gb_mean"] is not None else "Peak VRAM (GB): N/A\n")
        t.write(f"Throughput (tokens/s): {summary['throughput_toks_per_s_mean']:.2f} ± {summary['throughput_toks_per_s_std']:.2f}\n" if summary["throughput_toks_per_s_mean"] is not None else "Throughput (tokens/s): N/A\n")
        t.write("Notes: " + summary["notes"] + "\n")
        t.write("=" * 80 + "\n\n")

    json_out_path.parent.mkdir(parents=True, exist_ok=True)
    with json_out_path.open("w", encoding="utf-8") as jf:
        json.dump({"summary": summary, "per_run": per_run}, jf, indent=2, ensure_ascii=False)

    csv_out_path.parent.mkdir(parents=True, exist_ok=True)
    csv_header = "model,device,lat_mean_ms,lat_std_ms,peak_vram_gb,peak_vram_std,throughput_tps,input_tokens,output_tokens\n"
    if not csv_out_path.exists():
        csv_out_path.write_text(csv_header, encoding="utf-8")

    csv_vals = [
        summary.get("model", ""),
        summary.get("device", ""),
        f"{summary['latency_ms_mean']:.6f}" if summary["latency_ms_mean"] is not None else "",
        f"{summary['latency_ms_std']:.6f}" if summary["latency_ms_std"] is not None else "",
        f"{summary['peak_vram_gb_mean']:.6f}" if summary["peak_vram_gb_mean"] is not None else "",
        f"{summary['peak_vram_gb_std']:.6f}" if summary["peak_vram_gb_std"] is not None else "",
        f"{summary['throughput_toks_per_s_mean']:.6f}" if summary["throughput_toks_per_s_mean"] is not None else "",
        str(summary.get("input_tokens", "")),
        str(summary.get("output_tokens", "")),
    ]
    with csv_out_path.open("a", encoding="utf-8") as cf:
        cf.write(",".join(csv_vals) + "\n")

    lat_str = f"{summary['latency_ms_mean']:.1f}" if summary["latency_ms_mean"] is not None else "N/A"
    thr_str = f"{summary['throughput_toks_per_s_mean']:.1f}" if summary["throughput_toks_per_s_mean"] is not None else "N/A"
    latex_row = f"{summary['model']} & {summary['device']} & {lat_str} & {thr_str} \\\\"

    latex_out_path.parent.mkdir(parents=True, exist_ok=True)
    with latex_out_path.open("w", encoding="utf-8") as lf:
        lf.write("% LaTeX table row for Table VI\n")
        lf.write(latex_row + "\n")

    print("Wrote TXT summary to", txt_log_path.resolve())
    print("Wrote JSON detail to", json_out_path.resolve())
    print("Appended CSV row to", csv_out_path.resolve())
    print("Wrote LaTeX row to", latex_out_path.resolve())
    print("\nSummary:\n", json.dumps(summary, indent=2))
    print("\nLaTeX row:\n", latex_row)

    return {"summary": summary, "per_run": per_run, "latex_row": latex_row}

## Quick sanity checks

In [34]:
# Optional: count questions by task
task_counts = print_task_question_counts(DATA_DIR)

Found 488 episode/MCQ pairs in /home/jupyter-mferrag/data/6GBench/mcq_questions_only

Question count per task:
------------------------------------------------------------
 T1 | Intent Feasibility Assessment                 |   115
T10 | SLA Violation Prediction                      |   115
T11 | Preemptive Autonomy Downgrade                 |   119
T12 | Conservative Continuation Decision            |   107
T13 | Swarm-Level Slice Negotiation                 |   128
T14 | Scheduler Reconfiguration Adaptation          |   107
T15 | Decision Consistency under Replanning         |   129
T16 | Network-Exposed Compute Marketplace           |   155
T17 | Network-Knowledge RAG Augmentation            |   154
T18 | AI Agent Identity & Onboarding                |   157
T19 | AI Agent Interoperability & Federation        |   129
 T2 | Intent Conflict Resolution                    |   113
T20 | Agent-to-Agent Communication Management       |   105
T21 | Device-Network Task Offload Arbitration   

In [35]:
from pathlib import Path
import json
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_file = Path(f"sanity_check_{EVAL_MODEL}_{timestamp}.txt")

# Quick single-question sanity check
ensure_model_available(EVAL_MODEL)

pairs = load_episode_mcq_pairs(DATA_DIR)
episode_id, ep, mcq = pairs[0]
q = mcq["questions"][0]

messages = build_eval_messages(
    q["task_id"],
    summarize_episode_for_prompt(ep, SUMMARY_MAX_TURNS),
    q,
)

raw_text, debug_info = local_chat_debug(
    model=EVAL_MODEL,
    messages=messages,
    temperature=0.0,
    max_tokens=1000,
    seed=42,
)

parsed_answer = parse_mcq_answer_from_json(raw_text)

# Build text output
output_text = f"""
Raw model output:
{raw_text}

Parsed answer: {parsed_answer}

Debug info:
{json.dumps(debug_info, indent=2)}
"""

# print to notebook
print(output_text)

# save to file
with open(out_file, "w") as f:
    f.write(output_text)

print(f"\nSaved output to: {out_file.resolve()}")

Found 488 episode/MCQ pairs in /home/jupyter-mferrag/data/6GBench/mcq_questions_only

Raw model output:
{
  "answer": "A"
}

Parsed answer: A

Debug info:
{
  "model": "smollm2:135m-instruct-q4_0",
  "created_at": "2026-04-30T16:19:33.2186875Z",
  "done_reason": "stop",
  "latency_ms": 138.081653,
  "load_duration_ms": 25.732391,
  "prompt_eval_count": 2102,
  "prompt_eval_duration_ms": 38.442936,
  "eval_count": 11,
  "eval_duration_ms": 13.830799,
  "throughput_toks_per_s": 795.3264305265373,
  "used_max_new_tokens": 1000,
  "peak_vram_gb": 0.8896484375,
  "input_tokens": 2102,
  "output_tokens": 11
}


Saved output to: /home/jupyter-mferrag/Ollama/sanity_check_smollm2:135m-instruct-q4_0_20260430_161932.txt


## Full evaluation

In [36]:
# full evaluation
results = eval_model_on_dir(
     data_dir=DATA_DIR,
     model=EVAL_MODEL,
     max_pairs=None,
     seed_base=42,
 )
#
print("Overall accuracy:", results["overall_accuracy"])
print("Overall pass@k:", results["overall_passk"])
print("Total questions:", results["total_questions"])

Found 488 episode/MCQ pairs in /home/jupyter-mferrag/data/6GBench/mcq_questions_only
Total episodes: 488 | Total questions (all episodes): 3722 | Avg questions/episode: 7.63


Eval episodes:   0%|          | 0/488 [00:00<?, ?ep/s]

[running] episodes=1/488 | questions=9 | overall_acc=0.222
[running] episodes=2/488 | questions=11 | overall_acc=0.182
[running] episodes=3/488 | questions=13 | overall_acc=0.154
[running] episodes=4/488 | questions=23 | overall_acc=0.304
[running] episodes=5/488 | questions=32 | overall_acc=0.219
[running] episodes=6/488 | questions=42 | overall_acc=0.262
[running] episodes=7/488 | questions=52 | overall_acc=0.288
[running] episodes=8/488 | questions=64 | overall_acc=0.234
[running] episodes=9/488 | questions=74 | overall_acc=0.257
[running] episodes=10/488 | questions=76 | overall_acc=0.250
[running] episodes=11/488 | questions=84 | overall_acc=0.226
[running] episodes=12/488 | questions=86 | overall_acc=0.221
[running] episodes=13/488 | questions=87 | overall_acc=0.230
[running] episodes=14/488 | questions=97 | overall_acc=0.237
[running] episodes=15/488 | questions=99 | overall_acc=0.232
[running] episodes=16/488 | questions=106 | overall_acc=0.217
[running] episodes=17/488 | quest

In [37]:
import json
from datetime import datetime

out_dir = Path("results")
out_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

out_file = out_dir / f"6gbench_{EVAL_MODEL}_{timestamp}.json"

with open(out_file, "w") as f:
    json.dump(results, f, indent=2)

print("Saved results to:", out_file)

Saved results to: results/6gbench_smollm2:135m-instruct-q4_0_20260430_164454.json


In [38]:
unload_model(EVAL_MODEL)

Unloaded model from Ollama memory: smollm2:135m-instruct-q4_0
